# Generate Matching Images for Evaluation Samples (SD 1.5)

**Self-contained notebook — no repo upload needed.**

Upload only:
1. This notebook
2. `rq3_samples_extended.json` (same directory)

What it does:
- Generates 47 images with Stable Diffusion 1.5 for new AudioCaps samples
- Only baseline (24) + wrong_audio (23) conditions (image should match text)
- Skips wrong_image (23) — mismatched image is intentional
- Recomputes CLIP text-image similarity (st_i) and MSCI
- Saves updated JSON + images as a zip for download

**Runtime: ~4 min on RTX 6000**

## Step 1: Environment fixes + imports

University GPU servers often have broken peft/diffusers/transformers combos.
This cell patches everything before any imports happen.

In [ ]:
# ---- Must be the VERY FIRST cell executed (fresh kernel) ----
import subprocess, sys, types, importlib, importlib.util, shutil

# 0) Install diffusers 0.30.3 into clean isolated /tmp dir
shutil.rmtree("/tmp/sd_clean", ignore_errors=True)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--target=/tmp/sd_clean", "--no-deps",
    "diffusers==0.30.3",
])
sys.path.insert(0, "/tmp/sd_clean")
print("[0/4] diffusers 0.30.3 installed to /tmp/sd_clean")

# 1) Block peft (may be broken on system)
fake_peft = types.ModuleType('peft')
fake_peft.__version__ = '0.0.0'
fake_peft.__path__ = []
fake_peft.__spec__ = None
fake_peft.__file__ = None
sys.modules['peft'] = fake_peft
for sub in ['peft.tuners', 'peft.tuners.tuners_utils', 'peft.tuners.lora',
            'peft.config', 'peft.auto', 'peft.peft_model', 'peft.mapping',
            'peft.utils', 'peft.utils.constants']:
    sys.modules[sub] = types.ModuleType(sub)
_orig = importlib.util.find_spec
def _no_peft(name, *a, **k):
    return None if 'peft' in name else _orig(name, *a, **k)
importlib.util.find_spec = _no_peft
print("[1/4] peft blocked")

# 2) Patch any missing symbols (safe no-ops if they already exist)
import transformers, transformers.utils
for attr, val in [('HybridCache', type('X', (), {})), ('Cache', type('X', (), {})),
                  ('DynamicCache', type('X', (), {})), ('EncoderDecoderCache', type('X', (), {}))]:
    if not hasattr(transformers, attr):
        setattr(transformers, attr, val)
for attr, val in [('FLAX_WEIGHTS_NAME', 'flax_model.msgpack'),
                  ('SAFE_WEIGHTS_NAME', 'model.safetensors'),
                  ('WEIGHTS_NAME', 'pytorch_model.bin')]:
    if not hasattr(transformers.utils, attr):
        setattr(transformers.utils, attr, val)
print("[2/4] transformers patched")

# 3) Disable version checks
import transformers.utils.versions
transformers.utils.versions.require_version = lambda *a, **k: None
transformers.utils.versions.require_version_core = lambda *a, **k: None
print("[3/4] version checks disabled")

# 4) Import
import torch
from diffusers import StableDiffusionPipeline
from transformers import CLIPModel, CLIPProcessor
import diffusers
print(f"[4/4] imports OK\n")
print(f"torch:        {torch.__version__}")
print(f"diffusers:    {diffusers.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
print("\nReady!")

## Step 2: Load samples and select targets

In [ ]:
import json
from pathlib import Path

SAMPLES_CANDIDATES = [
    Path("rq3_samples_extended.json"),
    Path("runs/rq3/rq3_samples_extended.json"),
    Path("../runs/rq3/rq3_samples_extended.json"),
]
SAMPLES_PATH = None
for p in SAMPLES_CANDIDATES:
    if p.exists():
        SAMPLES_PATH = p
        break
assert SAMPLES_PATH is not None, (
    "rq3_samples_extended.json not found! Upload it to the same directory as this notebook."
)

with open(SAMPLES_PATH) as f:
    data = json.load(f)

samples = data["samples"]
print(f"Total samples: {len(samples)}")

targets = [
    s for s in samples
    if int(s["sample_id"][1:]) >= 31 and s["condition"] != "wrong_image"
]

baseline = [s for s in targets if s["condition"] == "baseline"]
wrong_audio = [s for s in targets if s["condition"] == "wrong_audio"]
skipped = sum(1 for s in samples if int(s["sample_id"][1:]) >= 31 and s["condition"] == "wrong_image")

print(f"\nTargets: {len(targets)} samples")
print(f"  baseline:    {len(baseline)}")
print(f"  wrong_audio: {len(wrong_audio)}")
print(f"  skipped (wrong_image): {skipped}")

for s in targets[:5]:
    print(f"  {s['sample_id']} ({s['condition']}): {s['prompt_text'][:70]}")

## Step 3: Generate images with Stable Diffusion 1.5

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
OUTPUT_DIR = Path("generated_eval_images")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Seed: {SEED}")
print(f"Output: {OUTPUT_DIR}/")

# safety_checker=None skips the NSFW filter model (broken on transformers 5.x)
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe = pipe.to(DEVICE)
print("SD 1.5 loaded.")

In [ ]:
from IPython.display import display, Image as IPImage

generated_paths = {}

for i, sample in enumerate(targets):
    sid = sample["sample_id"]
    out_path = OUTPUT_DIR / f"{sid}_{sample['condition']}.png"

    generator = torch.Generator(device="cpu").manual_seed(SEED)

    result = pipe(
        prompt=sample["prompt_text"],
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=generator,
    )
    result.images[0].save(str(out_path))
    generated_paths[sid] = str(out_path)

    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1}/{len(targets)}] {sid}: {sample['prompt_text'][:60]}")
        display(IPImage(filename=str(out_path), width=256))

print(f"\nDone! Generated {len(generated_paths)} images.")

In [ ]:
# Free GPU memory
del pipe
if torch.cuda.is_available():
    torch.cuda.empty_cache()
import gc; gc.collect()
print("SD pipeline unloaded.")

## Step 4: Recompute CLIP text-image similarity (st_i)

In [ ]:
import numpy as np
from PIL import Image

CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_model.eval()
print(f"CLIP loaded on {DEVICE}")


def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-12)
    b = b / (np.linalg.norm(b) + 1e-12)
    return float(np.clip(np.dot(a, b), -1.0, 1.0))


@torch.no_grad()
def clip_text_embedding(text):
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    feats = clip_model.get_text_features(**inputs)
    return feats[0].cpu().numpy().astype("float32")


@torch.no_grad()
def clip_image_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    feats = clip_model.get_image_features(**inputs)
    return feats[0].cpu().numpy().astype("float32")

In [ ]:
W_TI = 0.45  # text-image weight
W_TA = 0.45  # text-audio weight

results = []

for i, sample in enumerate(targets):
    sid = sample["sample_id"]
    img_path = generated_paths[sid]

    old_st_i = sample["st_i"]

    text_emb = clip_text_embedding(sample["prompt_text"])
    image_emb = clip_image_embedding(img_path)
    new_st_i = cosine_sim(text_emb, image_emb)

    sample["image_path"] = img_path
    sample["st_i"] = round(new_st_i, 4)
    sample["msci"] = round(W_TI * new_st_i + W_TA * sample["st_a"], 4)

    results.append({
        "sid": sid,
        "condition": sample["condition"],
        "old_st_i": old_st_i,
        "new_st_i": new_st_i,
        "delta": new_st_i - old_st_i,
    })

    if (i + 1) % 10 == 0:
        print(f"[{i+1}/{len(targets)}] {sid}: st_i {old_st_i:.3f} -> {new_st_i:.3f} ({new_st_i - old_st_i:+.3f})")

print(f"\nRecomputed st_i for {len(results)} samples.")

## Step 5: Summary and verification

In [ ]:
baseline_results = [r for r in results if r["condition"] == "baseline"]
wa_results = [r for r in results if r["condition"] == "wrong_audio"]

print("=" * 60)
print("BEFORE/AFTER st_i (text-image CLIP similarity)")
print("=" * 60)

if baseline_results:
    old_mean = np.mean([r["old_st_i"] for r in baseline_results])
    new_mean = np.mean([r["new_st_i"] for r in baseline_results])
    print(f"\nBaseline ({len(baseline_results)} samples):")
    print(f"  Before (retrieved): {old_mean:.3f}")
    print(f"  After  (SD 1.5):    {new_mean:.3f}")
    print(f"  Improvement:        {new_mean - old_mean:+.3f}")

if wa_results:
    old_mean = np.mean([r["old_st_i"] for r in wa_results])
    new_mean = np.mean([r["new_st_i"] for r in wa_results])
    print(f"\nWrong-audio ({len(wa_results)} samples):")
    print(f"  Before (retrieved): {old_mean:.3f}")
    print(f"  After  (SD 1.5):    {new_mean:.3f}")
    print(f"  Improvement:        {new_mean - old_mean:+.3f}")

orig_baseline = [
    s for s in data["samples"]
    if int(s["sample_id"][1:]) < 31 and s["condition"] == "baseline"
]
if orig_baseline:
    print(f"\nOriginal 30 baseline mean st_i: {np.mean([s['st_i'] for s in orig_baseline]):.3f}")

print("\n" + "=" * 60)
print("SPOT CHECK: 4 generated images")
print("=" * 60)
from IPython.display import display, Image as IPImage
for s in targets[:4]:
    p = Path(generated_paths[s["sample_id"]])
    print(f"\n{s['sample_id']} ({s['condition']}): st_i={s['st_i']:.3f}")
    print(f"  \"{s['prompt_text'][:80]}\"")
    display(IPImage(filename=str(p), width=300))

## Step 6: Save updated JSON and zip everything for download

In [ ]:
import shutil

target_map = {s["sample_id"]: s for s in targets}
for i, sample in enumerate(data["samples"]):
    if sample["sample_id"] in target_map:
        data["samples"][i] = target_map[sample["sample_id"]]

updated_json_path = Path("rq3_samples_extended_updated.json")
with open(updated_json_path, "w") as f:
    json.dump(data, f, indent=2)
print(f"Saved: {updated_json_path}")

zip_dir = Path("eval_images_package")
if zip_dir.exists():
    shutil.rmtree(zip_dir)
zip_dir.mkdir(exist_ok=True)

img_dir = zip_dir / "generated_eval_images"
img_dir.mkdir(exist_ok=True)
for sid, path in generated_paths.items():
    shutil.copy2(path, img_dir / Path(path).name)

shutil.copy2(updated_json_path, zip_dir / "rq3_samples_extended.json")

zip_path = shutil.make_archive("eval_images_package", "zip", ".", "eval_images_package")
print(f"\nDownload: {zip_path}")
print(f"  Contains: {len(generated_paths)} images + updated rq3_samples_extended.json")
print(f"\nAfter download, copy to your local project:")
print(f"  1. Images -> data/generated/eval_images/")
print(f"  2. rq3_samples_extended.json -> runs/rq3/rq3_samples_extended.json")

In [ ]:
try:
    from IPython.display import FileLink
    display(FileLink("eval_images_package.zip"))
except:
    print(f"Download manually: {Path('eval_images_package.zip').resolve()}")